In [118]:
import pandas as pd
import numpy as np
import requests
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 10)

In [119]:
soil = pd.read_csv("../../data/processed/soil_data_cleaned.csv")

crop = pd.read_csv("../../data/processed/crop_production_cleaned.csv")

coords = pd.read_csv("../../data/processed/district_registry_geocoded.csv")

In [120]:
print("="*60)
print("SOIL DATA")
print("="*60)
print(soil.shape)
display(soil.head())

print("="*60)
print("CROP DATA")
print("="*60)
print(crop.shape)
display(crop.head())

print("="*60)
print("COORDINATE DATA")
print("="*60)
print(coords.shape)
display(coords.head())

SOIL DATA
(10853209, 14)


,id,year,state_name,state_code,district_name,district_code,block_name,block_code,village_name,village_code,nutrient_type,nutrient_name,nutrient_level,value
0,3107023,2023-24,Uttarakhand,5,Udham Singh Nagar,56,Rudrapur,439,Malsi,55943,Micro,Copper,Deficient,0
1,3107024,2023-24,Uttarakhand,5,Udham Singh Nagar,56,Rudrapur,439,Malsi,55943,Micro,Copper,Sufficient,85
2,3107025,2023-24,Uttarakhand,5,Udham Singh Nagar,56,Rudrapur,439,Malsi,55943,Micro,Iron,Deficient,0
3,3107026,2023-24,Uttarakhand,5,Udham Singh Nagar,56,Rudrapur,439,Malsi,55943,Micro,Iron,Sufficient,85
4,3107027,2023-24,Uttarakhand,5,Udham Singh Nagar,56,Rudrapur,439,Malsi,55943,Micro,Manganese,Deficient,2


CROP DATA
(246091, 8)


,index,State_Name,District_Name,Crop_Year,Season,Crop,Area,Production
0,0,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Arecanut,1254.0,2000.0
1,1,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Other Kharif pulses,2.0,1.0
2,2,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Rice,102.0,321.0
3,3,Andaman and Nicobar Islands,NICOBARS,2000,Whole Year,Banana,176.0,641.0
4,4,Andaman and Nicobar Islands,NICOBARS,2000,Whole Year,Cashewnut,720.0,165.0


COORDINATE DATA
(652, 4)


,State_Name,District_Name,latitude,longitude
0,Andaman and Nicobar Islands,NICOBARS,9.150000,92.750000
1,Andaman and Nicobar Islands,NORTH AND MIDDLE ANDAMAN,12.611239,92.831654
2,Andaman and Nicobar Islands,SOUTH ANDAMANS,11.620000,92.730000
3,Andhra Pradesh,ANANTAPUR,14.678322,77.606504
4,Andhra Pradesh,CHITTOOR,13.325036,79.648061


In [121]:
print("SOIL COLUMNS")
print(soil.columns.tolist())

print()

print("CROP COLUMNS")
print(crop.columns.tolist())

print()

print("COORDINATE COLUMNS")
print(coords.columns.tolist())

SOIL COLUMNS
['id', 'year', 'state_name', 'state_code', 'district_name', 'district_code', 'block_name', 'block_code', 'village_name', 'village_code', 'nutrient_type', 'nutrient_name', 'nutrient_level', 'value']

CROP COLUMNS
['index', 'State_Name', 'District_Name', 'Crop_Year', 'Season', 'Crop', 'Area', 'Production']

COORDINATE COLUMNS
['State_Name', 'District_Name', 'latitude', 'longitude']


In [122]:
print("SOIL")

display(soil.isnull().sum())

print()

print("CROP")

display(crop.isnull().sum())

print()

print("COORDINATES")

display(coords.isnull().sum())

SOIL


id                0
year              0
state_name        0
state_code        0
district_name     0
                 ..
village_code      0
nutrient_type     0
nutrient_name     0
nutrient_level    0
value             0
Length: 14, dtype: int64


CROP


index               0
State_Name          0
District_Name       0
Crop_Year           0
Season              0
Crop                0
Area                0
Production       3730
dtype: int64


COORDINATES


State_Name       0
District_Name    0
latitude         0
longitude        0
dtype: int64

In [123]:
soil["nutrient_name"].value_counts()

nutrient_name
Organic Carbon             1122793
Phosphorus                 1122778
Potassium                  1122777
Nitrogen                   1122703
Soil Ph                    1122664
                            ...   
Iron                        748501
Sulphur                     748491
Copper                      748490
Boron                       748479
Electrical Conductivity     748475
Name: count, Length: 12, dtype: int64

In [124]:
soil["value"].describe()

count    1.085321e+07
mean     1.409020e+01
std      4.490956e+01
min      0.000000e+00
25%      0.000000e+00
50%      2.000000e+00
75%      1.100000e+01
max      1.020500e+04
Name: value, dtype: float64

In [125]:
# Keep only the required columns
soil = soil[
    [
        "state_name",
        "district_name",
        "nutrient_name",
        "value"
    ]
].copy()

print(soil.shape)
soil.head()

(10853209, 4)


,state_name,district_name,nutrient_name,value
0,Uttarakhand,Udham Singh Nagar,Copper,0
1,Uttarakhand,Udham Singh Nagar,Copper,85
2,Uttarakhand,Udham Singh Nagar,Iron,0
3,Uttarakhand,Udham Singh Nagar,Iron,85
4,Uttarakhand,Udham Singh Nagar,Manganese,2


In [126]:
soil.isnull().sum()


state_name       0
district_name    0
nutrient_name    0
value            0
dtype: int64

In [127]:
soil_avg = (
    soil
    .groupby(
        ["state_name", "district_name", "nutrient_name"],
        as_index=False
    )["value"]
    .mean()
)

soil_avg.head()

,state_name,district_name,nutrient_name,value
0,Andaman And Nicobar Islands,Nicobars,Boron,7.070796
1,Andaman And Nicobar Islands,Nicobars,Copper,7.070796
2,Andaman And Nicobar Islands,Nicobars,Electrical Conductivity,7.008772
3,Andaman And Nicobar Islands,Nicobars,Iron,7.070796
4,Andaman And Nicobar Islands,Nicobars,Manganese,7.070796


In [128]:
soil_profile = (
    soil_avg
    .pivot(
        index=["state_name", "district_name"],
        columns="nutrient_name",
        values="value"
    )
    .reset_index()
)

soil_profile.head()

nutrient_name,state_name,district_name,Boron,Copper,Electrical Conductivity,Iron,Manganese,Nitrogen,Organic Carbon,Phosphorus,Potassium,Soil Ph,Sulphur,Zinc
0,Andaman And Nicobar Islands,Nicobars,7.070796,7.070796,7.008772,7.070796,7.070796,4.727811,4.727811,4.727811,4.727811,4.727811,7.070796,7.070796
1,Andaman And Nicobar Islands,North And Middle Andaman,30.027523,30.022936,30.022936,30.027523,30.027523,20.018349,20.018349,20.018349,20.018349,20.018349,30.027523,30.027523
2,Andaman And Nicobar Islands,South Andamans,16.793814,16.793814,16.793814,16.793814,16.793814,11.195876,11.192440,11.195876,11.195876,11.195876,16.793814,16.793814
3,Andhra Pradesh,Alluri Sitharama Raju,4.039506,4.042222,3.995309,4.039506,4.043457,2.685761,2.670453,2.688889,2.690864,2.645267,4.039506,4.035062
4,Andhra Pradesh,Anakapalli,13.856935,13.548012,13.852085,13.529098,13.508729,9.231490,9.234400,9.246363,9.238927,9.229227,13.859845,13.441319


In [129]:
print("District Soil Profile Shape:")
print(soil_profile.shape)

print()

print("Columns:")
print(soil_profile.columns.tolist())

District Soil Profile Shape:
(738, 14)

Columns:
['state_name', 'district_name', 'Boron', 'Copper', 'Electrical Conductivity', 'Iron', 'Manganese', 'Nitrogen', 'Organic Carbon', 'Phosphorus', 'Potassium', 'Soil Ph', 'Sulphur', 'Zinc']


In [130]:
soil_profile.to_csv(
    "../../data/processed/district_soil_profile.csv",
    index=False
)

print("✅ District soil profile saved successfully.")

✅ District soil profile saved successfully.


In [131]:
# Crop dataset
crop = crop.rename(columns={
    "State_Name": "state_name",
    "District_Name": "district_name",
    "Crop_Year": "year",
    "Crop": "crop",
    "Season": "season",
    "Area": "area",
    "Production": "production",
    "yield": "yield"
})

# Remove extra spaces and convert to uppercase for reliable matching
crop["state_name"] = crop["state_name"].str.strip().str.upper()
crop["district_name"] = crop["district_name"].str.strip().str.upper()

soil_profile["state_name"] = soil_profile["state_name"].str.strip().str.upper()
soil_profile["district_name"] = soil_profile["district_name"].str.strip().str.upper()

print("Crop:", crop.shape)
print("Soil:", soil_profile.shape)

Crop: (246091, 8)
Soil: (738, 14)


In [132]:
master = crop.merge(
    soil_profile,
    on=["state_name", "district_name"],
    how="left"
)

print("Master Shape:", master.shape)

master.head()

Master Shape: (246091, 20)


,index,state_name,district_name,year,season,crop,area,production,Boron,Copper,Electrical Conductivity,Iron,Manganese,Nitrogen,Organic Carbon,Phosphorus,Potassium,Soil Ph,Sulphur,Zinc
0,0,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,2000,Kharif,Arecanut,1254.0,2000.0,7.070796,7.070796,7.008772,7.070796,7.070796,4.727811,4.727811,4.727811,4.727811,4.727811,7.070796,7.070796
1,1,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,2000,Kharif,Other Kharif pulses,2.0,1.0,7.070796,7.070796,7.008772,7.070796,7.070796,4.727811,4.727811,4.727811,4.727811,4.727811,7.070796,7.070796
2,2,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,2000,Kharif,Rice,102.0,321.0,7.070796,7.070796,7.008772,7.070796,7.070796,4.727811,4.727811,4.727811,4.727811,4.727811,7.070796,7.070796
3,3,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,2000,Whole Year,Banana,176.0,641.0,7.070796,7.070796,7.008772,7.070796,7.070796,4.727811,4.727811,4.727811,4.727811,4.727811,7.070796,7.070796
4,4,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,2000,Whole Year,Cashewnut,720.0,165.0,7.070796,7.070796,7.008772,7.070796,7.070796,4.727811,4.727811,4.727811,4.727811,4.727811,7.070796,7.070796


In [133]:
print("Missing values after merge:")

master[
    [
        "Nitrogen",
        "Phosphorus",
        "Potassium",
        "Organic Carbon",
        "Soil Ph"
    ]
].isnull().sum()

Missing values after merge:


Nitrogen          38405
Phosphorus        38405
Potassium         38405
Organic Carbon    38405
Soil Ph           38405
dtype: int64

In [134]:
print("Total crop records:", len(master))

matched = master["Nitrogen"].notna().sum()

print("Matched records:", matched)
print("Coverage:", round((matched / len(master)) * 100, 2), "%")

Total crop records: 246091
Matched records: 207686
Coverage: 84.39 %


In [135]:
print(sorted(crop["year"].unique()))

[np.int64(1997), np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015)]


In [136]:
unmatched = master[master["Nitrogen"].isna()].copy()

print("Unmatched Records:", len(unmatched))

unmatched_districts = (
    unmatched[["state_name", "district_name"]]
    .drop_duplicates()
    .sort_values(["state_name", "district_name"])
)

print("Unique Unmatched Districts:", len(unmatched_districts))

unmatched_districts.head(20)

Unmatched Records: 38405
Unique Unmatched Districts: 94


,state_name,district_name
203,ANDHRA PRADESH,ANANTAPUR
3232,ANDHRA PRADESH,KADAPA
6328,ANDHRA PRADESH,SPSR NELLORE
7699,ANDHRA PRADESH,VISAKHAPATANAM
21468,ASSAM,KARIMGANJ
...,...,...
66801,HARYANA,GURGAON
69109,HARYANA,MEWAT
72697,HIMACHAL PRADESH,LAHUL AND SPITI
74057,JAMMU AND KASHMIR,BADGAM


In [137]:
unmatched = master[master["Nitrogen"].isna()].copy()

print("Unmatched Records:", len(unmatched))

Unmatched Records: 38405


In [138]:
unmatched_districts = (
    unmatched[["state_name", "district_name"]]
    .drop_duplicates()
    .sort_values(["state_name", "district_name"])
    .reset_index(drop=True)
)

print("Unique unmatched districts:", len(unmatched_districts))

unmatched_districts.head(20)

Unique unmatched districts: 94


,state_name,district_name
0,ANDHRA PRADESH,ANANTAPUR
1,ANDHRA PRADESH,KADAPA
2,ANDHRA PRADESH,SPSR NELLORE
3,ANDHRA PRADESH,VISAKHAPATANAM
4,ASSAM,KARIMGANJ
...,...,...
15,HARYANA,GURGAON
16,HARYANA,MEWAT
17,HIMACHAL PRADESH,LAHUL AND SPITI
18,JAMMU AND KASHMIR,BADGAM


In [139]:
soil_profile[
    soil_profile["district_name"].str.upper() == "ADILABAD"
]

nutrient_name,state_name,district_name,Boron,Copper,Electrical Conductivity,Iron,Manganese,Nitrogen,Organic Carbon,Phosphorus,Potassium,Soil Ph,Sulphur,Zinc
588,TELANGANA,ADILABAD,111.7,50.98,111.96,50.98,50.98,74.64,74.64,74.64,74.626667,74.626667,111.76,50.98


In [140]:
soil_profile["district_name"].value_counts().head(20)

district_name
BILASPUR                    2
HAMIRPUR                    2
PRATAPGARH                  2
NICOBARS                    1
NORTH AND MIDDLE ANDAMAN    1
                           ..
GUNTUR                      1
KAKINADA                    1
KRISHNA                     1
KURNOOL                     1
NANDYAL                     1
Name: count, Length: 20, dtype: int64

In [141]:
# Soil profile using only district name
soil_by_district = soil_profile.drop(columns=["state_name"])

# Try matching only the unmatched records
unmatched_recovered = unmatched.drop(
    columns=soil_profile.columns[2:],  # remove empty soil columns
    errors="ignore"
).merge(
    soil_by_district,
    on="district_name",
    how="left"
)

print(unmatched_recovered.shape)

print("Recovered Records:",
      unmatched_recovered["Nitrogen"].notna().sum())

print("Still Unmatched:",
      unmatched_recovered["Nitrogen"].isna().sum())

(38405, 20)
Recovered Records: 1400
Still Unmatched: 37005


In [142]:
# Find district names that appear exactly once
unique_districts = (
    soil_profile["district_name"]
    .value_counts()
)

unique_districts = unique_districts[unique_districts == 1].index

print("Unique district names:", len(unique_districts))

Unique district names: 732


In [143]:
# Keep only districts that occur exactly once
soil_unique = soil_profile[
    soil_profile["district_name"].isin(unique_districts)
].copy()

print(soil_unique.shape)

(732, 14)


In [144]:
# Remove the empty soil columns from unmatched records
soil_columns = [
    "Boron",
    "Copper",
    "Electrical Conductivity",
    "Iron",
    "Manganese",
    "Nitrogen",
    "Organic Carbon",
    "Phosphorus",
    "Potassium",
    "Soil Ph",
    "Sulphur",
    "Zinc"
]

unmatched_clean = unmatched.drop(columns=soil_columns, errors="ignore")

# Recover using only unique district names
recovered = unmatched_clean.merge(
    soil_unique.drop(columns=["state_name"]),
    on="district_name",
    how="left"
)

print("Recovered Shape:", recovered.shape)
print("Recovered Records:", recovered["Nitrogen"].notna().sum())
print("Still Missing:", recovered["Nitrogen"].isna().sum())

Recovered Shape: (38405, 20)
Recovered Records: 1400
Still Missing: 37005


In [145]:
# Keep only the records that were recovered successfully
recovered_success = recovered[recovered["Nitrogen"].notna()].copy()

# Keep the original matched records
master_matched = master[master["Nitrogen"].notna()].copy()

# Combine them
master_final = pd.concat(
    [master_matched, recovered_success],
    ignore_index=True
)

print("Final Dataset Shape:", master_final.shape)
print("Coverage:", round(len(master_final) / len(crop) * 100, 2), "%")

Final Dataset Shape: (209086, 20)
Coverage: 84.96 %


In [146]:
master_final.to_csv(
    "../../data/processed/master_soil_crop_v2.csv",
    index=False
)

## Weather


In [147]:
weather_requests = (
    master_final[
        ["state_name", "district_name", "year"]
    ]
    .drop_duplicates()
    .sort_values(
        ["state_name", "district_name", "year"]
    )
    .reset_index(drop=True)
)

print(weather_requests.shape)

weather_requests.head()

(8467, 3)


,state_name,district_name,year
0,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,2000
1,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,2001
2,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,2002
3,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,2003
4,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,2004


In [148]:
coords = pd.read_csv("../../data/processed/district_registry_geocoded.csv")

print(coords.columns.tolist())

print(coords.head())

['State_Name', 'District_Name', 'latitude', 'longitude']
                    State_Name             District_Name   latitude  longitude
0  Andaman and Nicobar Islands                  NICOBARS   9.150000  92.750000
1  Andaman and Nicobar Islands  NORTH AND MIDDLE ANDAMAN  12.611239  92.831654
2  Andaman and Nicobar Islands            SOUTH ANDAMANS  11.620000  92.730000
3               Andhra Pradesh                 ANANTAPUR  14.678322  77.606504
4               Andhra Pradesh                  CHITTOOR  13.325036  79.648061


In [149]:
print(coords[["latitude", "longitude"]].isna().sum())

print()

print("Rows with coordinates:",
      coords.dropna(subset=["latitude", "longitude"]).shape[0])

print("Total rows:",
      len(coords))

latitude     0
longitude    0
dtype: int64

Rows with coordinates: 652
Total rows: 652


In [150]:
import os

for file in os.listdir("../../data/processed"):
    print(file)

crop_production_cleaned.csv
district_crop_database.csv
district_crop_database_standardized.csv
district_manual_mapping.csv
district_matching_report.csv
district_registry.csv
district_registry_geocoded.csv
district_soil_database.csv
district_soil_profile.csv
master_soil_crop_v2.csv
missing_district_coordinates.csv
soil_data_cleaned.csv
weather_districts.csv
weather_yearly.csv


In [151]:
gadm = pd.read_csv(
    "../../data/processed/district_registry_geocoded.csv"
)

print(gadm.shape)
print(gadm.columns.tolist())

gadm.head()

(652, 4)
['State_Name', 'District_Name', 'latitude', 'longitude']


,State_Name,District_Name,latitude,longitude
0,Andaman and Nicobar Islands,NICOBARS,9.150000,92.750000
1,Andaman and Nicobar Islands,NORTH AND MIDDLE ANDAMAN,12.611239,92.831654
2,Andaman and Nicobar Islands,SOUTH ANDAMANS,11.620000,92.730000
3,Andhra Pradesh,ANANTAPUR,14.678322,77.606504
4,Andhra Pradesh,CHITTOOR,13.325036,79.648061


In [152]:
print("master_final columns:")
print(master_final.columns.tolist())

master_final columns:
['index', 'state_name', 'district_name', 'year', 'season', 'crop', 'area', 'production', 'Boron', 'Copper', 'Electrical Conductivity', 'Iron', 'Manganese', 'Nitrogen', 'Organic Carbon', 'Phosphorus', 'Potassium', 'Soil Ph', 'Sulphur', 'Zinc']


In [153]:
print("gadm columns:")
print(gadm.columns.tolist())

gadm columns:
['State_Name', 'District_Name', 'latitude', 'longitude']


In [154]:
master_final["State_Name"] = (
    master_final["state_name"]
    .astype(str)
    .str.strip()
    .str.upper()
)

master_final["District_Name"] = (
    master_final["district_name"]
    .astype(str)
    .str.strip()
    .str.upper()
)

In [155]:
# Standardize names
gadm["State_Name"] = gadm["State_Name"].str.strip().str.upper()
gadm["District_Name"] = gadm["District_Name"].str.strip().str.upper()

# Merge with our master dataset
weather_base = master_final.merge(
    gadm,
    on=["State_Name", "District_Name"],
    how="left"
)

print("Master Shape:", weather_base.shape)

print("Rows with Coordinates:",
      weather_base.dropna(subset=["latitude", "longitude"]).shape[0])

print("Coverage:",
      round(
          weather_base["latitude"].notna().mean() * 100,
          2
      ),
      "%"
)

Master Shape: (209086, 24)
Rows with Coordinates: 209086
Coverage: 100.0 %


In [156]:
weather_districts = (
    weather_base[
        ["state_name", "district_name", "latitude", "longitude"]
    ]
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
)

In [157]:
# Create one request per district

weather_districts = (
    weather_base[
        [
            "state_name",
            "district_name",
            "latitude",
            "longitude"
        ]
    ]
    .dropna(subset=["latitude", "longitude"])
    .drop_duplicates()
    .sort_values(["state_name", "district_name"])
    .reset_index(drop=True)
)

print("Unique districts:", len(weather_districts))

weather_districts.head()

Unique districts: 563


,state_name,district_name,latitude,longitude
0,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,9.150000,92.750000
1,ANDAMAN AND NICOBAR ISLANDS,NORTH AND MIDDLE ANDAMAN,12.611239,92.831654
2,ANDAMAN AND NICOBAR ISLANDS,SOUTH ANDAMANS,11.620000,92.730000
3,ANDHRA PRADESH,CHITTOOR,13.325036,79.648061
4,ANDHRA PRADESH,EAST GODAVARI,16.995664,81.715438


In [158]:
weather_districts.to_csv(
    "../../data/processed/weather_districts.csv",
    index=False
)

print("Saved successfully!")

Saved successfully!


In [159]:
import requests
import pandas as pd
from pathlib import Path

In [178]:
district = weather_districts.iloc[0]

lat = district["latitude"]
lon = district["longitude"]

print("District :", district["district_name"])
print("State    :", district["state_name"])
print("Latitude :", lat)
print("Longitude:", lon)

District : NICOBARS
State    : ANDAMAN AND NICOBAR ISLANDS
Latitude : 9.15
Longitude: 92.75


In [179]:
import requests
import pandas as pd

lat = 9.15
lon = 92.75

url = (
    "https://power.larc.nasa.gov/api/temporal/daily/point"
    f"?parameters=T2M,PRECTOTCORR,RH2M,WS2M,ALLSKY_SFC_SW_DWN"
    f"&community=AG"
    f"&latitude={lat}"
    f"&longitude={lon}"
    f"&start=19970101"
    f"&end=20201231"
    f"&format=JSON"
)

response = requests.get(url, timeout=60)

print("Status Code:", response.status_code)

data = response.json()

print(data.keys())

Status Code: 200
dict_keys(['type', 'geometry', 'properties', 'header', 'messages', 'parameters', 'times'])


In [176]:
import requests
import pandas as pd
from pathlib import Path
from time import sleep

# -----------------------------------
# Paths
# -----------------------------------

REGISTRY_PATH = Path(
    "../../data/processed/district_registry_geocoded.csv"
)

OUTPUT_DIR = Path(
    "../../data/weather"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# -----------------------------------
# Load district registry
# -----------------------------------

districts = pd.read_csv(REGISTRY_PATH)

# Clean names
districts["State_Name"] = (
    districts["State_Name"]
    .astype(str)
    .str.strip()
)

districts["District_Name"] = (
    districts["District_Name"]
    .astype(str)
    .str.strip()
)

print("Total districts:", len(districts))

# -----------------------------------
# NASA POWER download
# -----------------------------------

for index, row in districts.iterrows():

    state = row["State_Name"]
    district = row["District_Name"]

    lat = row["latitude"]
    lon = row["longitude"]

    # Filename
    filename = (
        f"{state}_{district}"
        .upper()
        .replace(" ", "_")
        .replace("/", "_")
        .replace("(", "")
        .replace(")", "")
        .replace("&", "AND")
        + ".csv"
    )

    output_file = OUTPUT_DIR / filename

    # --------------------------------
    # Skip already downloaded files
    # --------------------------------

    if output_file.exists():

        print(
            f"[{index + 1}/{len(districts)}] "
            f"Already exists: {district}"
        )

        continue

    print(
        f"[{index + 1}/{len(districts)}] "
        f"Downloading: {state} - {district}"
    )

    # --------------------------------
    # NASA POWER URL
    # --------------------------------

    url = (
        "https://power.larc.nasa.gov/api/temporal/daily/point"
        "?parameters=T2M,PRECTOTCORR,RH2M,WS2M,ALLSKY_SFC_SW_DWN"
        "&community=AG"
        f"&latitude={lat}"
        f"&longitude={lon}"
        "&start=19970101"
        "&end=20201231"
        "&format=JSON"
    )

    try:

        response = requests.get(
            url,
            timeout=60
        )

        response.raise_for_status()

        data = response.json()

        # --------------------------------
        # Extract parameters
        # --------------------------------

        parameters = data[
            "properties"
        ][
            "parameter"
        ]

        weather_df = pd.DataFrame(
            parameters
        )

        weather_df.index.name = "date"

        weather_df.reset_index(
            inplace=True
        )

        # --------------------------------
        # Add location information
        # --------------------------------

        weather_df["State_Name"] = state
        weather_df["District_Name"] = district
        weather_df["latitude"] = lat
        weather_df["longitude"] = lon

        # --------------------------------
        # Save
        # --------------------------------

        weather_df.to_csv(
            output_file,
            index=False
        )

        print(
            f"   ✓ Saved {len(weather_df)} rows"
        )

    except Exception as e:

        print(
            f"   ✗ Failed: {district}"
        )

        print(
            f"     Error: {e}"
        )



print("\nDownload process completed.")

Total districts: 652
[1/652] Already exists: NICOBARS
[2/652] Already exists: NORTH AND MIDDLE ANDAMAN
[3/652] Already exists: SOUTH ANDAMANS
[4/652] Already exists: ANANTAPUR
[5/652] Already exists: CHITTOOR
[6/652] Already exists: EAST GODAVARI
[7/652] Already exists: GUNTUR
[8/652] Already exists: KADAPA
[9/652] Already exists: KRISHNA
[10/652] Already exists: KURNOOL
[11/652] Already exists: PRAKASAM
[12/652] Already exists: SPSR NELLORE
[13/652] Already exists: SRIKAKULAM
[14/652] Already exists: VISAKHAPATANAM
[15/652] Already exists: VIZIANAGARAM
[16/652] Already exists: WEST GODAVARI
[17/652] Already exists: ANJAW
[18/652] Already exists: CHANGLANG
[19/652] Already exists: DIBANG VALLEY
[20/652] Already exists: EAST KAMENG
[21/652] Already exists: EAST SIANG
[22/652] Already exists: KURUNG KUMEY
[23/652] Already exists: LOHIT
[24/652] Already exists: LONGDING
[25/652] Already exists: LOWER DIBANG VALLEY
[26/652] Already exists: LOWER SUBANSIRI
[27/652] Already exists: NAMSAI
[2

In [180]:
weather_df["state_name"] = district["state_name"]
weather_df["district_name"] = district["district_name"]

In [181]:
weather_files = list(
    OUTPUT_DIR.glob("*.csv")
)

print(
    "Weather files created:",
    len(weather_files)
)

Weather files created: 652


In [182]:
parameters = data["properties"]["parameter"]

weather_test = pd.DataFrame(parameters)

weather_test.index.name = "date"
weather_test.reset_index(inplace=True)

print("Shape:", weather_test.shape)
print("\nColumns:")
print(weather_test.columns.tolist())

print("\nFirst 5 rows:")
print(weather_test.head())

Shape: (8766, 6)

Columns:
['date', 'T2M', 'PRECTOTCORR', 'RH2M', 'WS2M', 'ALLSKY_SFC_SW_DWN']

First 5 rows:
       date    T2M  PRECTOTCORR   RH2M  WS2M  ALLSKY_SFC_SW_DWN
0  19970101  26.22         0.17  71.52  5.34              19.07
1  19970102  26.58         0.02  70.50  5.16              20.84
2  19970103  26.81         0.00  67.64  4.48              21.26
3  19970104  26.82         0.02  71.07  4.80              18.26
4  19970105  26.37         0.06  73.69  5.18              16.90


In [183]:
from pathlib import Path

OUTPUT_DIR = Path("../../data/weather")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Weather folder:", OUTPUT_DIR.resolve())

Weather folder: D:\VS CODE SAVES\NTCC\KrishKalp\KrishiKalp-Smart-Crop-Recommendation-website-\data\weather


In [184]:
url = (
    "https://power.larc.nasa.gov/api/temporal/daily/point"
    f"?parameters=T2M,PRECTOTCORR,RH2M,WS2M,ALLSKY_SFC_SW_DWN"
    f"&community=AG"
    f"&latitude={lat}"
    f"&longitude={lon}"
    f"&start=19970101"
    f"&end=20201231"
    f"&format=JSON"
)

response = requests.get(url, timeout=60)

print("Status Code:", response.status_code)

Status Code: 200


In [185]:
data = response.json()

print(data.keys())

dict_keys(['type', 'geometry', 'properties', 'header', 'messages', 'parameters', 'times'])


In [186]:
print(data["properties"].keys())

dict_keys(['parameter'])


In [187]:
print(data["properties"]["parameter"].keys())

dict_keys(['T2M', 'PRECTOTCORR', 'RH2M', 'WS2M', 'ALLSKY_SFC_SW_DWN'])


In [188]:
t2m = data["properties"]["parameter"]["T2M"]

print(type(t2m))
print("Number of days:", len(t2m))

# Show first 5 entries
list(t2m.items())[:5]

<class 'dict'>
Number of days: 8766


[('19970101', 26.22),
 ('19970102', 26.58),
 ('19970103', 26.81),
 ('19970104', 26.82),
 ('19970105', 26.37)]

In [189]:
t2m_df = (
    pd.DataFrame(
        t2m.items(),
        columns=["date", "temperature"]
    )
)

t2m_df.head()

,date,temperature
0,19970101,26.22
1,19970102,26.58
2,19970103,26.81
3,19970104,26.82
4,19970105,26.37


In [190]:
weather_df = pd.DataFrame({
    "date": list(data["properties"]["parameter"]["T2M"].keys()),
    "temperature": list(data["properties"]["parameter"]["T2M"].values()),
    "rainfall": list(data["properties"]["parameter"]["PRECTOTCORR"].values()),
    "humidity": list(data["properties"]["parameter"]["RH2M"].values()),
    "wind_speed": list(data["properties"]["parameter"]["WS2M"].values()),
    "solar_radiation": list(data["properties"]["parameter"]["ALLSKY_SFC_SW_DWN"].values()),
})

weather_df.head()



,date,temperature,rainfall,humidity,wind_speed,solar_radiation
0,19970101,26.22,0.17,71.52,5.34,19.07
1,19970102,26.58,0.02,70.50,5.16,20.84
2,19970103,26.81,0.00,67.64,4.48,21.26
3,19970104,26.82,0.02,71.07,4.80,18.26
4,19970105,26.37,0.06,73.69,5.18,16.90


In [191]:
weather_df["state_name"] = district["state_name"]
weather_df["district_name"] = district["district_name"]

# Reorder columns
weather_df = weather_df[
    [
        "state_name",
        "district_name",
        "date",
        "temperature",
        "rainfall",
        "humidity",
        "wind_speed",
        "solar_radiation"
    ]
]

weather_df.head()

,state_name,district_name,date,temperature,rainfall,humidity,wind_speed,solar_radiation
0,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,19970101,26.22,0.17,71.52,5.34,19.07
1,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,19970102,26.58,0.02,70.50,5.16,20.84
2,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,19970103,26.81,0.00,67.64,4.48,21.26
3,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,19970104,26.82,0.02,71.07,4.80,18.26
4,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,19970105,26.37,0.06,73.69,5.18,16.90


In [192]:
weather_df["date"] = pd.to_datetime(
    weather_df["date"],
    format="%Y%m%d"
)

weather_df["year"] = weather_df["date"].dt.year

weather_df.head()

,state_name,district_name,date,temperature,rainfall,humidity,wind_speed,solar_radiation,year
0,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,1997-01-01,26.22,0.17,71.52,5.34,19.07,1997
1,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,1997-01-02,26.58,0.02,70.50,5.16,20.84,1997
2,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,1997-01-03,26.81,0.00,67.64,4.48,21.26,1997
3,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,1997-01-04,26.82,0.02,71.07,4.80,18.26,1997
4,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,1997-01-05,26.37,0.06,73.69,5.18,16.90,1997


In [193]:
yearly_weather = (
    weather_df.groupby(
        ["state_name", "district_name", "year"],
        as_index=False
    )
    .agg({
        "temperature": "mean",
        "rainfall": "sum",
        "humidity": "mean",
        "wind_speed": "mean",
        "solar_radiation": "mean"
    })
)

yearly_weather.head()

,state_name,district_name,year,temperature,rainfall,humidity,wind_speed,solar_radiation
0,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,1997,27.548849,1440.00,80.644932,4.814548,19.315781
1,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,1998,28.019781,2002.28,81.466137,4.985205,17.853644
2,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,1999,27.460877,2016.93,82.039151,5.304137,18.164493
3,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,2000,27.495109,2096.57,81.762404,5.256749,18.200765
4,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,2001,27.684466,2040.81,81.582603,5.188192,18.138466


In [194]:
print("Shape:", yearly_weather.shape)
print("Years:", yearly_weather["year"].min(), "-", yearly_weather["year"].max())

Shape: (24, 8)
Years: 1997 - 2020


In [195]:
def download_weather(latitude, longitude):
    """
    Download daily weather data (1997-2020)
    from NASA POWER API.
    """

    url = (
        "https://power.larc.nasa.gov/api/temporal/daily/point"
        f"?parameters=T2M,PRECTOTCORR,RH2M,WS2M,ALLSKY_SFC_SW_DWN"
        f"&community=AG"
        f"&latitude={latitude}"
        f"&longitude={longitude}"
        f"&start=19970101"
        f"&end=20201231"
        f"&format=JSON"
    )

    response = requests.get(url, timeout=60)

    response.raise_for_status()

    return response.json()

In [196]:
def parse_weather(data, state_name, district_name):
    """
    Convert NASA POWER JSON to a daily weather DataFrame.
    """

    weather_df = pd.DataFrame({
        "date": list(data["properties"]["parameter"]["T2M"].keys()),
        "temperature": list(data["properties"]["parameter"]["T2M"].values()),
        "rainfall": list(data["properties"]["parameter"]["PRECTOTCORR"].values()),
        "humidity": list(data["properties"]["parameter"]["RH2M"].values()),
        "wind_speed": list(data["properties"]["parameter"]["WS2M"].values()),
        "solar_radiation": list(data["properties"]["parameter"]["ALLSKY_SFC_SW_DWN"].values()),
    })

    weather_df["state_name"] = state_name
    weather_df["district_name"] = district_name

    weather_df["date"] = pd.to_datetime(
        weather_df["date"],
        format="%Y%m%d"
    )

    weather_df["year"] = weather_df["date"].dt.year

    return weather_df

In [197]:
def aggregate_weather(weather_df):
    """
    Aggregate daily weather data into yearly weather statistics.
    """

    yearly_weather = (
        weather_df
        .groupby(
            ["state_name", "district_name", "year"],
            as_index=False
        )
        .agg({
            "temperature": "mean",
            "rainfall": "sum",
            "humidity": "mean",
            "wind_speed": "mean",
            "solar_radiation": "mean"
        })
    )

    return yearly_weather

In [198]:
data = download_weather(lat, lon)

daily_weather = parse_weather(
    data,
    district["state_name"],
    district["district_name"]
)

yearly_weather = aggregate_weather(daily_weather)

print(yearly_weather.shape)

yearly_weather.head()

(24, 8)


,state_name,district_name,year,temperature,rainfall,humidity,wind_speed,solar_radiation
0,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,1997,27.548849,1440.00,80.644932,4.814548,19.315781
1,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,1998,28.019781,2002.28,81.466137,4.985205,17.853644
2,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,1999,27.460877,2016.93,82.039151,5.304137,18.164493
3,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,2000,27.495109,2096.57,81.762404,5.256749,18.200765
4,ANDAMAN AND NICOBAR ISLANDS,NICOBARS,2001,27.684466,2040.81,81.582603,5.188192,18.138466


In [199]:
from pathlib import Path

OUTPUT_DIR = Path("../../data/weather")

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(OUTPUT_DIR)

..\..\data\weather


In [200]:
all_weather = []

for _, row in weather_districts.head(3).iterrows():

    print(f"Processing: {row['district_name']}")

    data = download_weather(
        row["latitude"],
        row["longitude"]
    )

    daily_weather = parse_weather(
        data,
        row["state_name"],
        row["district_name"]
    )

    yearly_weather = aggregate_weather(daily_weather)

    all_weather.append(yearly_weather)

print("\nCompleted!")

Processing: NICOBARS
Processing: NORTH AND MIDDLE ANDAMAN
Processing: SOUTH ANDAMANS

Completed!


In [201]:
for _, row in weather_districts.iterrows():

    filename = (
        row["state_name"] + "_" + row["district_name"]
    ).replace(" ", "_").replace("/", "_") + ".csv"

    filepath = OUTPUT_DIR / filename

    if filepath.exists():
        print(f"Skipping: {filename}")
        continue

    try:

        print(f"Processing: {row['district_name']}")

        data = download_weather(
            row["Latitude"],
            row["Longitude"]
        )

        daily_weather = parse_weather(
            data,
            row["state_name"],
            row["district_name"]
        )

        yearly_weather = aggregate_weather(daily_weather)

        yearly_weather.to_csv(
            filepath,
            index=False
        )

        print(f"Saved: {filename}")

    except Exception as e:

        print(f"Failed: {filename}")
        print(e)

print("\nAll districts processed!")

Skipping: ANDAMAN_AND_NICOBAR_ISLANDS_NICOBARS.csv
Skipping: ANDAMAN_AND_NICOBAR_ISLANDS_NORTH_AND_MIDDLE_ANDAMAN.csv
Skipping: ANDAMAN_AND_NICOBAR_ISLANDS_SOUTH_ANDAMANS.csv
Skipping: ANDHRA_PRADESH_CHITTOOR.csv
Skipping: ANDHRA_PRADESH_EAST_GODAVARI.csv
Skipping: ANDHRA_PRADESH_GUNTUR.csv
Skipping: ANDHRA_PRADESH_KRISHNA.csv
Skipping: ANDHRA_PRADESH_KURNOOL.csv
Skipping: ANDHRA_PRADESH_PRAKASAM.csv
Skipping: ANDHRA_PRADESH_SRIKAKULAM.csv
Skipping: ANDHRA_PRADESH_VIZIANAGARAM.csv
Skipping: ANDHRA_PRADESH_WEST_GODAVARI.csv
Skipping: ARUNACHAL_PRADESH_ANJAW.csv
Skipping: ARUNACHAL_PRADESH_CHANGLANG.csv
Skipping: ARUNACHAL_PRADESH_DIBANG_VALLEY.csv
Skipping: ARUNACHAL_PRADESH_EAST_KAMENG.csv
Skipping: ARUNACHAL_PRADESH_EAST_SIANG.csv
Skipping: ARUNACHAL_PRADESH_KURUNG_KUMEY.csv
Skipping: ARUNACHAL_PRADESH_LOHIT.csv
Skipping: ARUNACHAL_PRADESH_LONGDING.csv
Skipping: ARUNACHAL_PRADESH_LOWER_DIBANG_VALLEY.csv
Skipping: ARUNACHAL_PRADESH_LOWER_SUBANSIRI.csv
Skipping: ARUNACHAL_PRADESH_NAMSAI

In [202]:
from pathlib import Path
import pandas as pd

weather_files = list(OUTPUT_DIR.glob("*.csv"))

print("Total files:", len(weather_files))

weather_df = pd.concat(
    [pd.read_csv(file) for file in weather_files],
    ignore_index=True
)

print(weather_df.shape)

weather_df.head()

Total files: 652
(5715432, 10)


,date,T2M,PRECTOTCORR,RH2M,WS2M,ALLSKY_SFC_SW_DWN,State_Name,District_Name,latitude,longitude
0,19970101,26.22,0.17,71.52,5.34,19.07,Andaman and Nicobar Islands,NICOBARS,9.15,92.75
1,19970102,26.58,0.02,70.50,5.16,20.84,Andaman and Nicobar Islands,NICOBARS,9.15,92.75
2,19970103,26.81,0.00,67.64,4.48,21.26,Andaman and Nicobar Islands,NICOBARS,9.15,92.75
3,19970104,26.82,0.02,71.07,4.80,18.26,Andaman and Nicobar Islands,NICOBARS,9.15,92.75
4,19970105,26.37,0.06,73.69,5.18,16.90,Andaman and Nicobar Islands,NICOBARS,9.15,92.75


In [203]:
downloaded = {
    f.stem.replace("_", " ")
    for f in weather_files
}

expected = {
    (row["state_name"] + "_" + row["district_name"])
    .replace(" ", "_")
    .replace("/", "_")
    for _, row in weather_districts.iterrows()
}

missing = expected - {f.stem for f in weather_files}

print("Missing districts:", len(missing))

for district in sorted(missing):
    print(district)

Missing districts: 1
BIHAR_KAIMUR_(BHABUA)


In [204]:
print("Unique district names:", weather_districts["district_name"].nunique())
print("Total districts:", len(weather_districts))

duplicates = weather_districts[
    weather_districts.duplicated("district_name", keep=False)
].sort_values("district_name")

duplicates

Unique district names: 557
Total districts: 563


,state_name,district_name,latitude,longitude
58,BIHAR,AURANGABAD,24.803320,84.411020
294,MAHARASHTRA,AURANGABAD,19.877263,75.339024
95,CHHATTISGARH,BALRAMPUR,23.628258,83.481092
488,UTTAR PRADESH,BALRAMPUR,27.447699,82.395624
98,CHHATTISGARH,BIJAPUR,18.793568,80.815939
...,...,...,...,...
161,HIMACHAL PRADESH,BILASPUR,31.338096,76.761163
163,HIMACHAL PRADESH,HAMIRPUR,31.653987,76.540363
508,UTTAR PRADESH,HAMIRPUR,25.750000,80.000000
428,RAJASTHAN,PRATAPGARH,24.023222,74.625028


In [206]:
weather_districts = (
    weather_base[
        ["state_name", "district_name", "latitude", "longitude"]
    ]
    .dropna(subset=["latitude", "longitude"])
    .drop_duplicates()
    .sort_values(["state_name", "district_name"])
    .reset_index(drop=True)
)

# Keep only one coordinate per district
weather_districts = (
    weather_districts
    .drop_duplicates(
        subset=["state_name", "district_name"],
        keep="first"
    )
    .reset_index(drop=True)
)

print("Unique districts:", len(weather_districts))

Unique districts: 563


In [207]:
weather_df.to_csv(
    "../../data/processed/weather_yearly.csv",
    index=False
)

print("Saved successfully!")

Saved successfully!


In [208]:
print(weather_yearly.columns.tolist())
print(weather_yearly.shape)
print(weather_yearly.head())

['State_Name', 'District_Name', 'year', 'temperature', 'rainfall', 'humidity', 'wind_speed', 'solar_radiation']
(15648, 8)
                    State_Name District_Name  year  temperature  rainfall  \
0  Andaman and Nicobar Islands      NICOBARS  1997    27.548849   1440.00   
1  Andaman and Nicobar Islands      NICOBARS  1998    28.019781   2002.28   
2  Andaman and Nicobar Islands      NICOBARS  1999    27.460877   2016.93   
3  Andaman and Nicobar Islands      NICOBARS  2000    27.495109   2096.57   
4  Andaman and Nicobar Islands      NICOBARS  2001    27.684466   2040.81   

    humidity  wind_speed  solar_radiation  
0  80.644932    4.814548        19.315781  
1  81.466137    4.985205        17.853644  
2  82.039151    5.304137        18.164493  
3  81.762404    5.256749        18.200765  
4  81.582603    5.188192        18.138466  


In [211]:
print("Shape:", weather_yearly.shape)

print(weather_yearly.columns.tolist())

print(weather_yearly.head())

Shape: (15648, 8)
['State_Name', 'District_Name', 'year', 'temperature', 'rainfall', 'humidity', 'wind_speed', 'solar_radiation']
                    State_Name District_Name  year  temperature  rainfall  \
0  Andaman and Nicobar Islands      NICOBARS  1997    27.548849   1440.00   
1  Andaman and Nicobar Islands      NICOBARS  1998    28.019781   2002.28   
2  Andaman and Nicobar Islands      NICOBARS  1999    27.460877   2016.93   
3  Andaman and Nicobar Islands      NICOBARS  2000    27.495109   2096.57   
4  Andaman and Nicobar Islands      NICOBARS  2001    27.684466   2040.81   

    humidity  wind_speed  solar_radiation  
0  80.644932    4.814548        19.315781  
1  81.466137    4.985205        17.853644  
2  82.039151    5.304137        18.164493  
3  81.762404    5.256749        18.200765  
4  81.582603    5.188192        18.138466  


In [212]:
import pandas as pd

# Load the CURRENT daily weather data
weather_daily = pd.read_csv(
    "../../data/processed/weather_yearly.csv"
)

print("Daily weather shape:", weather_daily.shape)

# Create year
weather_daily["year"] = (
    weather_daily["date"]
    .astype(str)
    .str[:4]
    .astype(int)
)

# Aggregate daily -> yearly
weather_yearly = (
    weather_daily
    .groupby(
        ["State_Name", "District_Name", "year"],
        as_index=False
    )
    .agg(
        temperature=("T2M", "mean"),
        rainfall=("PRECTOTCORR", "sum"),
        humidity=("RH2M", "mean"),
        wind_speed=("WS2M", "mean"),
        solar_radiation=("ALLSKY_SFC_SW_DWN", "mean")
    )
)

print("Yearly weather shape:", weather_yearly.shape)
print(weather_yearly.columns.tolist())
print(weather_yearly.head())

Daily weather shape: (5715432, 10)
Yearly weather shape: (15648, 8)
['State_Name', 'District_Name', 'year', 'temperature', 'rainfall', 'humidity', 'wind_speed', 'solar_radiation']
                    State_Name District_Name  year  temperature  rainfall  \
0  Andaman and Nicobar Islands      NICOBARS  1997    27.548849   1440.00   
1  Andaman and Nicobar Islands      NICOBARS  1998    28.019781   2002.28   
2  Andaman and Nicobar Islands      NICOBARS  1999    27.460877   2016.93   
3  Andaman and Nicobar Islands      NICOBARS  2000    27.495109   2096.57   
4  Andaman and Nicobar Islands      NICOBARS  2001    27.684466   2040.81   

    humidity  wind_speed  solar_radiation  
0  80.644932    4.814548        19.315781  
1  81.466137    4.985205        17.853644  
2  82.039151    5.304137        18.164493  
3  81.762404    5.256749        18.200765  
4  81.582603    5.188192        18.138466  


In [214]:
print("Shape:", weather_yearly.shape)
print(weather_yearly.columns.tolist())
print(weather_yearly.head())

Shape: (15648, 8)
['State_Name', 'District_Name', 'year', 'temperature', 'rainfall', 'humidity', 'wind_speed', 'solar_radiation']
                    State_Name District_Name  year  temperature  rainfall  \
0  Andaman and Nicobar Islands      NICOBARS  1997    27.548849   1440.00   
1  Andaman and Nicobar Islands      NICOBARS  1998    28.019781   2002.28   
2  Andaman and Nicobar Islands      NICOBARS  1999    27.460877   2016.93   
3  Andaman and Nicobar Islands      NICOBARS  2000    27.495109   2096.57   
4  Andaman and Nicobar Islands      NICOBARS  2001    27.684466   2040.81   

    humidity  wind_speed  solar_radiation  
0  80.644932    4.814548        19.315781  
1  81.466137    4.985205        17.853644  
2  82.039151    5.304137        18.164493  
3  81.762404    5.256749        18.200765  
4  81.582603    5.188192        18.138466  


In [221]:
# Clean merge keys

master_final["state_name"] = (
    master_final["state_name"]
    .astype(str)
    .str.strip()
    .str.upper()
)

master_final["district_name"] = (
    master_final["district_name"]
    .astype(str)
    .str.strip()
    .str.upper()
)

weather_yearly["State_Name"] = (
    weather_yearly["State_Name"]
    .astype(str)
    .str.strip()
    .str.upper()
)

weather_yearly["District_Name"] = (
    weather_yearly["District_Name"]
    .astype(str)
    .str.strip()
    .str.upper()
)

In [222]:
master_dataset = master_final.merge(
    weather_yearly,
    left_on=["state_name", "district_name", "year"],
    right_on=["State_Name", "District_Name", "year"],
    how="left"
)

In [223]:
print(master_dataset.shape)
print(master_dataset.columns.tolist())

(209086, 29)
['index', 'state_name', 'district_name', 'year', 'season', 'crop', 'area', 'production', 'Boron', 'Copper', 'Electrical Conductivity', 'Iron', 'Manganese', 'Nitrogen', 'Organic Carbon', 'Phosphorus', 'Potassium', 'Soil Ph', 'Sulphur', 'Zinc', 'State_Name_x', 'District_Name_x', 'State_Name_y', 'District_Name_y', 'temperature', 'rainfall', 'humidity', 'wind_speed', 'solar_radiation']


In [224]:
print(
    "State x vs State y:",
    (master_dataset["State_Name_x"] == master_dataset["State_Name_y"]).mean()
)

print(
    "District x vs District y:",
    (master_dataset["District_Name_x"] == master_dataset["District_Name_y"]).mean()
)

State x vs State y: 1.0
District x vs District y: 1.0


In [225]:
master_dataset.drop(
    columns=[
        "State_Name_x",
        "District_Name_x",
        "State_Name_y",
        "District_Name_y"
    ],
    inplace=True
)

In [226]:
print(
    master_dataset[
        [
            "temperature",
            "rainfall",
            "humidity",
            "wind_speed",
            "solar_radiation"
        ]
    ].isna().sum()
)

temperature        0
rainfall           0
humidity           0
wind_speed         0
solar_radiation    0
dtype: int64


In [227]:
master_dataset.drop(
    columns=["index"],
    inplace=True
)

In [228]:
print("Shape:", master_dataset.shape)

print("\nColumns:")
print(master_dataset.columns.tolist())

Shape: (209086, 24)

Columns:
['state_name', 'district_name', 'year', 'season', 'crop', 'area', 'production', 'Boron', 'Copper', 'Electrical Conductivity', 'Iron', 'Manganese', 'Nitrogen', 'Organic Carbon', 'Phosphorus', 'Potassium', 'Soil Ph', 'Sulphur', 'Zinc', 'temperature', 'rainfall', 'humidity', 'wind_speed', 'solar_radiation']


In [229]:
print(
    "Duplicate rows:",
    master_dataset.duplicated().sum()
)

Duplicate rows: 0


In [230]:
required_features = [
    "state_name",
    "district_name",
    "season",
    "Boron",
    "Copper",
    "Electrical Conductivity",
    "Iron",
    "Manganese",
    "Nitrogen",
    "Organic Carbon",
    "Phosphorus",
    "Potassium",
    "Soil Ph",
    "Sulphur",
    "Zinc",
    "temperature",
    "rainfall",
    "humidity",
    "wind_speed",
    "solar_radiation"
]

missing_features = [
    col for col in required_features
    if col not in master_dataset.columns
]

print("Missing model features:", missing_features)

Missing model features: []


In [231]:
master_dataset.to_csv(
    "../../data/processed/master_dataset.csv",
    index=False
)

print("master_dataset.csv saved successfully.")

master_dataset.csv saved successfully.


In [232]:
from pathlib import Path

path = Path("../../data/processed/master_dataset.csv")

print("File:", path.resolve())
print("Exists:", path.exists())
print("Size (MB):", round(path.stat().st_size / (1024 * 1024), 2))

File: D:\VS CODE SAVES\NTCC\KrishKalp\KrishiKalp-Smart-Crop-Recommendation-website-\data\processed\master_dataset.csv
Exists: True
Size (MB): 70.25


In [233]:
print("Final shape:", master_dataset.shape)

print("\nMissing values:")
print(master_dataset.isna().sum())

print("\nDuplicate rows:")
print(master_dataset.duplicated().sum())

print("\nUnique states:", master_dataset["state_name"].nunique())
print("Unique districts:", master_dataset["district_name"].nunique())
print("Unique crops:", master_dataset["crop"].nunique())
print("Unique years:", master_dataset["year"].nunique())
print("Unique seasons:", master_dataset["season"].nunique())

Final shape: (209086, 24)

Missing values:
state_name         0
district_name      0
year               0
season             0
crop               0
                  ..
temperature        0
rainfall           0
humidity           0
wind_speed         0
solar_radiation    0
Length: 24, dtype: int64

Duplicate rows:
0

Unique states: 30
Unique districts: 557
Unique crops: 123
Unique years: 19
Unique seasons: 6


In [235]:
import pandas as pd

crop_df = pd.read_csv(
    "../../data/raw/crop_production.csv"
)

print("Original crop production:", crop_df.shape)
print("Master final:", master_final.shape)
print("Final master:", master_dataset.shape)

Original crop production: (246091, 8)
Master final: (209086, 22)
Final master: (209086, 24)


In [237]:
crop_keys = crop_df[
    [
        "State_Name",
        "District_Name",
        "Crop_Year",
        "Season",
        "Crop"
    ]
].copy()

crop_keys.columns = [
    "state_name",
    "district_name",
    "year",
    "season",
    "crop"
]

# Normalize crop dataset keys
for col in ["state_name", "district_name", "season", "crop"]:
    crop_keys[col] = (
        crop_keys[col]
        .astype(str)
        .str.strip()
        .str.upper()
    )

# Normalize master keys
master_keys = master_dataset[
    [
        "state_name",
        "district_name",
        "year",
        "season",
        "crop"
    ]
].drop_duplicates().copy()

for col in ["state_name", "district_name", "season", "crop"]:
    master_keys[col] = (
        master_keys[col]
        .astype(str)
        .str.strip()
        .str.upper()
    )

# Make sure year has the same type
crop_keys["year"] = pd.to_numeric(
    crop_keys["year"],
    errors="coerce"
)

master_keys["year"] = pd.to_numeric(
    master_keys["year"],
    errors="coerce"
)

comparison = crop_keys.merge(
    master_keys,
    on=[
        "state_name",
        "district_name",
        "year",
        "season",
        "crop"
    ],
    how="left",
    indicator=True
)

print(comparison["_merge"].value_counts())

_merge
both          209086
left_only      37005
right_only         0
Name: count, dtype: int64


In [238]:
master_dataset.to_csv(
    "../../data/processed/master_dataset.csv",
    index=False
)

print("Master dataset saved successfully.")

Master dataset saved successfully.


In [239]:
check = pd.read_csv(
    "../../data/processed/master_dataset.csv"
)

print("Shape:", check.shape)
print("Missing values:", check.isna().sum().sum())
print("Duplicates:", check.duplicated().sum())

Shape: (209086, 24)
Missing values: 3405
Duplicates: 0


In [240]:
print(check.isna().sum())

state_name         0
district_name      0
year               0
season             0
crop               0
                  ..
temperature        0
rainfall           0
humidity           0
wind_speed         0
solar_radiation    0
Length: 24, dtype: int64


In [241]:
missing_rows = check[check.isna().any(axis=1)]

print("Rows with missing values:", len(missing_rows))
print(missing_rows.head())

Rows with missing values: 3405
                      state_name district_name  year       season  \
46   ANDAMAN AND NICOBAR ISLANDS      NICOBARS  2005  Whole Year    
51   ANDAMAN AND NICOBAR ISLANDS      NICOBARS  2005  Whole Year    
365               ANDHRA PRADESH      CHITTOOR  2001  Rabi          
529               ANDHRA PRADESH      CHITTOOR  2004  Rabi          
631               ANDHRA PRADESH      CHITTOOR  2007  Kharif        

                  crop     area  production      Boron     Copper  \
46            Arecanut   795.67         NaN   7.070796   7.070796   
51        Dry chillies    17.00         NaN   7.070796   7.070796   
365              Wheat     4.00         NaN  12.343402  12.296900   
529              Wheat     2.00         NaN  12.343402  12.296900   
631  Moong(Green Gram)  1000.00         NaN  12.343402  12.296900   

     Electrical Conductivity       Iron  Manganese  Nitrogen  Organic Carbon  \
46                  7.008772   7.070796   7.070796  4.72781

In [242]:
missing = check.isna().sum()

print(missing[missing > 0])

production    3405
dtype: int64
